# 1 · Your first tank

In this notebook you write a battle bot, fight the demo bots, find out why it
misses, and fix it with a little geometry. No viewer needed: battles run right
here, headless, in a fraction of a second.

**What you need:** Python basics (classes, `if`, functions). **Time:** 20 minutes.

Run the cells from top to bottom (Shift+Enter).

In [1]:
from terrabots.battle import BattleAction, BattleAgent, BattleConfig, BattleObservation, BattleRunner
from terrabots.battle.geometry import heading_to, normalize_angle, project
from terrabots.agents.sitting_duck import SittingDuck
from terrabots.agents.spin_bot import SpinBot
from terrabots.agents.walls import Walls
from terrabots.agents.tracker import Tracker

## A bot is one class

Every tick the engine calls `decide(obs)`. `obs` is what your tank knows right
now (position, headings, energy, what its radar saw); you return a
`BattleAction`: how fast to drive, how far to turn body, gun and radar, and how
hard to fire.

Headings are in degrees like on a compass: 0 = north, 90 = east. Distances are
in cells; the arena is 48 × 48 cells by default.

The radar sits on the gun, so a bot that just sweeps its gun around also sweeps
its radar. When the radar sees an enemy, we turn the gun towards it and fire.

In [2]:
class MyTank(BattleAgent):
    """Sweeps its gun until the radar sees someone, then aims and fires."""

    def decide(self, obs: BattleObservation) -> BattleAction:
        if not obs.scanned_bots:
            return BattleAction(velocity=0.3, body_turn=4, gun_turn=20)   # search

        enemy = obs.scanned_bots[0]
        aim = normalize_angle(enemy.bearing - obs.gun_heading)   # how far to turn the gun
        fire = 2.0 if abs(aim) < 5 and obs.gun_heat == 0 else 0.0
        return BattleAction(velocity=0.3, body_turn=4, gun_turn=aim, fire=fire)

## Fight

`BattleRunner` plays the rounds. The `seed` makes the battle repeatable: run the
cell twice and you get the same result, which is what makes fair comparisons
possible later. `time_budget_ms=None` switches off the per-turn time limit so
the result doesn't depend on how busy your laptop is.

In [3]:
def fight(bot, opponent, rounds=10, seed=1):
    config = BattleConfig(rounds=rounds, seed=seed, terrain="flat")
    return BattleRunner(config, [(bot, 1), (opponent, 1)], time_budget_ms=None).run()

print(fight(MyTank, SittingDuck).table())

rank  name                       total  survival    bullet     ram   1st   2nd   3rd    hit%
--------------------------------------------------------------------------------------------
   1  MyTank                      1800       500      1000       0    10     0     0    66.0
   2  SittingDuck                    0         0         0       0     0    10     0     0.0


Against a target that never moves, MyTank wins every round. Now
against `Walls`, which drives along the edges of the arena:

In [4]:
print(fight(MyTank, Walls).table())

rank  name                       total  survival    bullet     ram   1st   2nd   3rd    hit%
--------------------------------------------------------------------------------------------
   1  Walls                        811       450       226       0    10     0     0    16.0
   2  MyTank                       204         0       200       2     0    10     0     4.0


## Why it misses

Look at the `hit%` column. MyTank aims at where the enemy **is**. But a bullet
needs time to fly (power 2 flies 14 px = 0.875 cells per tick), and while it
flies the enemy keeps driving. At 15 cells that's 17 ticks, enough for a tank to
move several cells.

The fix is called **linear targeting**: assume the enemy keeps its heading and
speed, and aim where it will be when the bullet arrives. That point depends on
the flight time, which depends on the distance to that point, so we iterate a
few times until it settles.

In [5]:
class LeadingTank(MyTank):
    """MyTank with linear targeting: aims where the enemy will be."""

    POWER = 2.0

    def decide(self, obs: BattleObservation) -> BattleAction:
        if not obs.scanned_bots:
            return BattleAction(velocity=0.3, body_turn=4, gun_turn=20)

        enemy = obs.scanned_bots[0]
        ex, ey = project(obs.x, obs.y, enemy.bearing, enemy.distance)   # where it is now
        speed = self.rules.bullet_speed(self.POWER)                      # cells per tick
        px, py = ex, ey
        for _ in range(10):                                              # where it will be
            ticks = ((px - obs.x) ** 2 + (py - obs.y) ** 2) ** 0.5 / speed
            px, py = project(ex, ey, enemy.heading, enemy.velocity * ticks)

        aim = normalize_angle(heading_to(obs.x, obs.y, px, py) - obs.gun_heading)
        fire = self.POWER if abs(aim) < 3 and obs.gun_heat == 0 else 0.0
        return BattleAction(velocity=0.3, body_turn=4, gun_turn=aim, fire=fire)

print(fight(LeadingTank, Walls).table())

rank  name                       total  survival    bullet     ram   1st   2nd   3rd    hit%
--------------------------------------------------------------------------------------------
   1  LeadingTank                 1549       400       925       3     9     1     0    47.0
   2  Walls                        588        50       491       4     1     9     0    24.0


## Measure, don't guess

One battle can be luck. Run both bots against a few opponents over several seeds
and compare the hit rate:

In [6]:
import pandas as pd

rows = []
for bot in (MyTank, LeadingTank):
    for opponent in (SittingDuck, Walls, Tracker, SpinBot):
        for seed in range(3):
            r = next(r for r in fight(bot, opponent, seed=seed).rows if r["name"] == bot.__name__)
            rows.append({"bot": bot.__name__, "opponent": opponent.__name__, "seed": seed,
                         "hit_rate": r["hit_rate"], "firsts": r["firsts"]})

df = pd.DataFrame(rows)
df.pivot_table(index="opponent", columns="bot", values=["hit_rate", "firsts"]).round(2)

firsts           hit_rate       
bot         LeadingTank MyTank LeadingTank MyTank
opponent                                         
SittingDuck        7.67  10.00        0.94   0.70
SpinBot            1.67   5.33        0.14   0.17
Tracker            7.00   8.00        0.69   0.54
Walls              9.33   1.67        0.49   0.09

Linear targeting pays off against bots that drive straight: Walls and
Tracker. Against SpinBot it doesn't help: a bot driving in circles is never
where a straight line predicts. That's your first exercise.

## Exercises

1. **Circular targeting.** Remember the enemy's heading from the last scan; the
   change per tick is its turn rate. Predict along a circle instead of a line.
2. **Bullet power.** Strong bullets do more damage but fly slower and cost more
   energy. Fire harder when close, softer when far. Does the hit rate go up?
3. **Move!** MyTank drives in a slow circle and is easy to hit. Try driving
   perpendicular to the enemy (strafing) and reversing now and then.

## Watch your bot

Save your class in a file, e.g. `bots/my_tank.py` (start from
`bots/first_tank.py`), then run `terrabots-battle` and pick it in the setup
screen. Use `self.log("...")` in `decide()` and press **C** in the viewer to
read it live. The [battle guide](../docs/battle-guide.md) explains the rules,
physics and the classic strategies.